In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-06-15 11:48:37 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Evolución vinculación wompi primer registro

In [2]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_hist_vinc_wompi_vinc_primer_regi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_hist_vinc_wompi_vinc_primer_regi STORED AS PARQUET AS WITH news AS
  (SELECT periodo,
          num_vinc AS num_vinc_new,
          cast(left(cast(periodo AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(periodo AS STRING), 2) AS int) AS mes,
          1 AS secuencia
   FROM proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist
   WHERE tipo_cliente = 'nuevos' ),
                                                                             news_out AS
  (SELECT periodo,
          num_vinc_new,
          sum(num_vinc_new) OVER (PARTITION BY YEAR
                                  ORDER BY YEAR, mes) AS num_vinc_new_cumsum_ym,
          sum(num_vinc_new) OVER (PARTITION BY secuencia
                                  ORDER BY periodo) AS num_vinc_new_cumsum
   FROM news),
                                                                             olds AS
  (SELECT periodo,
          num_vinc AS num_vinc_old
   FROM proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist
   WHERE tipo_cliente = 'viejos' ),
                                                                             alls AS
  (SELECT periodo,
          num_vinc AS num_vinc_all
   FROM proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist
   WHERE tipo_cliente = 'todos' )
SELECT n.periodo,
       n.num_vinc_new,
       n.num_vinc_new_cumsum_ym,
       n.num_vinc_new_cumsum,
       o.num_vinc_old,
       a.num_vinc_all
FROM news_out AS n
LEFT JOIN olds AS o ON n.periodo = o.periodo
LEFT JOIN alls AS a ON n.periodo = a.periodo
ORDER BY periodo DESC
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_hist_vinc_wompi_vinc_primer_regi;"""
helper.ejecutar_consulta(sql_compute)

2026-06-15 11:48:39 - [INFO] - Transcurrido: 1781542120, Tiempo de Refresco = 1000


------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 1/1 DROP ....mdo_hist_vinc_wompi_vinc_primer_regi   finalizado   11:48:40 AM     00:00.8 
------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------
  i   tipo                   nombre                     estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 2/2 CREATE ....mdo_hist_vinc_wompi_vinc_primer_regi   finalizado   11:48:41 AM     00:01.5 
--------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------

# Uso de vinculados wompi primer registro

In [3]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_hist_vinc_uso_wompi_vinc_primer_regi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_hist_vinc_uso_wompi_vinc_primer_regi STORED AS PARQUET AS WITH news AS
  (SELECT periodo,
          count(*) AS num_vinc_new_uso_cumsum_ym
   FROM proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist
   WHERE tipo_cliente = 'nuevos'
   GROUP BY 1),
                                                                                 olds AS
  (SELECT periodo,
          count(*) AS num_vinc_old_uso_cumsum_ym
   FROM proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist
   WHERE tipo_cliente = 'viejos'
   GROUP BY 1),
                                                                                 alls AS
  (SELECT periodo,
          count(*) AS num_vinc_all_uso_cumsum_ym
   FROM proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist
   WHERE tipo_cliente = 'todos'
   GROUP BY 1)
SELECT n.periodo,
       n.num_vinc_new_uso_cumsum_ym,
       o.num_vinc_old_uso_cumsum_ym,
       a.num_vinc_all_uso_cumsum_ym
FROM news AS n
LEFT JOIN olds AS o ON n.periodo = o.periodo
LEFT JOIN alls AS a ON n.periodo = a.periodo
ORDER BY n.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_hist_vinc_uso_wompi_vinc_primer_regi;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 4/4    DROP ..._hist_vinc_uso_wompi_vinc_primer_regi   finalizado   11:48:44 AM     00:00.6 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 5/5  CREATE ..._hist_vinc_uso_wompi_vinc_primer_regi   finalizado   11:48:44 AM     00:01.7 
---------------------------------------------------------------------------------------------
------------------------------------------------------------

# Tabla resultado

In [4]:
# sql = """
# WITH outcome1 AS
#   (SELECT a.fecha_ym,
#           a.num_vinc_new,
#           a.num_vinc_cumsum,
#           a.num_vinc_new_cumsum_ym,
#           nvl(b.num_vinc_new_uso_cumsum_ym, 0) AS num_vinc_new_uso_cumsum_ym,
#           round(nvl(b.num_vinc_new_uso_cumsum_ym, 0)/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
#           nvl(c.num_vinc_old_uso_cumsum_ym, 0) AS num_vinc_old_uso_cumsum_ym,
#           nvl(d.num_vinc_all_uso_cumsum_ym, 0) AS num_vinc_all_uso_cumsum_ym,
#           round(nvl(d.num_vinc_all_uso_cumsum_ym, 0)/a.num_vinc_cumsum, 4) AS num_vinc_all_prop_uso,
#           left(cast(a.fecha_ym AS string), 4) AS YEAR,
#           right(cast(a.fecha_ym AS string), 2) AS mes
#    FROM proceso.mdo_aceptacion_comercios_vinc AS a
#    LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_new_uso AS b ON a.fecha_ym = b.fecha_ym
#    LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_old_uso AS c ON a.fecha_ym = c.fecha_ym
#    LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_all_uso AS d ON a.fecha_ym = d.fecha_ym),
#      outcome2 AS
#   (SELECT fecha_ym,
#           num_vinc_cumsum AS num_vinc_old,
#           cast(cast(YEAR AS int) + 1 AS string) AS YEAR
#    FROM outcome1
#    WHERE mes = '12')
# SELECT a.fecha_ym,
#        CONCAT(a.YEAR, '/', a.mes, '/', '01') AS fecha_ym2,
#        a.num_vinc_new,
#        a.num_vinc_new_cumsum_ym,
#        a.num_vinc_new_uso_cumsum_ym,
#        a.num_vinc_new_prop_uso,
#        b.num_vinc_old,
#        a.num_vinc_old_uso_cumsum_ym,
#        round(a.num_vinc_old_uso_cumsum_ym/b.num_vinc_old, 4) AS num_vinc_old_prop_uso,
#        a.num_vinc_cumsum,
#        a.num_vinc_all_uso_cumsum_ym,
#        a.num_vinc_all_prop_uso
# FROM outcome1 AS a
# LEFT JOIN outcome2 AS b ON a.year = b.year
# WHERE a.fecha_ym BETWEEN 202201 AND 202511
# ORDER BY a.fecha_ym DESC;
# """
# # print(sql)
# df_outcome = helper.obtener_dataframe(sql)
# df_outcome

In [5]:
sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_hist_vinc_y_uso_wompi_vinc_primer_regi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso_vdm.mdo_hist_vinc_y_uso_wompi_vinc_primer_regi STORED AS PARQUET AS
SELECT a.periodo,
       concat(cast(a.periodo as string), '01') AS fecha_ymd2,
       a.num_vinc_new,
       a.num_vinc_new_cumsum_ym,
       a.num_vinc_new_cumsum,
       a.num_vinc_old,
       a.num_vinc_all,
       b.num_vinc_new_uso_cumsum_ym,
       b.num_vinc_old_uso_cumsum_ym,
       b.num_vinc_all_uso_cumsum_ym,
       round(b.num_vinc_new_uso_cumsum_ym/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
       round(b.num_vinc_old_uso_cumsum_ym/a.num_vinc_old, 4) AS num_vinc_old_prop_uso,
       round(b.num_vinc_all_uso_cumsum_ym/a.num_vinc_all, 4) AS num_vinc_all_prop_uso
FROM proceso.mdo_hist_vinc_wompi_vinc_primer_regi AS a
LEFT JOIN proceso.mdo_hist_vinc_uso_wompi_vinc_primer_regi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_hist_vinc_y_uso_wompi_vinc_primer_regi;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 7/7    DROP ...ist_vinc_y_uso_wompi_vinc_primer_regi   finalizado   11:48:47 AM     00:00.6 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 8/8  CREATE ...ist_vinc_y_uso_wompi_vinc_primer_regi   finalizado   11:48:48 AM     00:01.1 
---------------------------------------------------------------------------------------------
------------------------------------------------------------

In [6]:
sql = """
SELECT a.periodo,
       a.fecha_ymd2,
       a.num_vinc_new,
       a.num_vinc_new_cumsum_ym,
       a.num_vinc_new_cumsum,
       a.num_vinc_old,
       a.num_vinc_all,
       a.num_vinc_new_uso_cumsum_ym,
       a.num_vinc_old_uso_cumsum_ym,
       a.num_vinc_all_uso_cumsum_ym,
       a.num_vinc_new_prop_uso,
       a.num_vinc_old_prop_uso,
       a.num_vinc_all_prop_uso
FROM proceso_vdm.mdo_hist_vinc_y_uso_wompi_vinc_primer_regi AS a
ORDER BY a.periodo DESC;
"""
df_outcome = helper.obtener_dataframe(sql)
df_outcome

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 10/10 DATAFRAME                                           descargando   11:48:50 AM             

2026-06-15 11:48:52 - [INFO] - 53 filas, 13 columnas, 00:01.8 consultando, 00:00.4 descargando, 00:00.0 convirtiendo


 10/10 DATAFRAME                                            finalizado   11:48:50 AM     00:02.5 
-------------------------------------------------------------------------------------------------


,periodo,fecha_ymd2,num_vinc_new,num_vinc_new_cumsum_ym,num_vinc_new_cumsum,num_vinc_old,num_vinc_all,num_vinc_new_uso_cumsum_ym,num_vinc_old_uso_cumsum_ym,num_vinc_all_uso_cumsum_ym,num_vinc_new_prop_uso,num_vinc_old_prop_uso,num_vinc_all_prop_uso
0,202605.0,20260501,7220,32553,121308,112313,144866,1209,19409,27048,0.0371,0.1728,0.1867
1,202604.0,20260401,7816,25333,114088,112313,137646,1120,18262,23595,0.0442,0.1626,0.1714
2,202603.0,20260301,7124,17517,106272,112313,129830,974,16942,20174,0.0556,0.1508,0.1554
3,202602.0,20260201,5692,10393,99148,112313,122706,718,15033,16694,0.0691,0.1338,0.1360
4,202601.0,20260101,4701,4701,93456,112313,117014,560,12275,12835,0.1191,0.1093,0.1097
5,202512.0,20251201,4245,40793,88755,71520,112313,553,17377,28209,0.0136,0.2430,0.2512
6,202511.0,20251101,4654,36548,84510,71520,108068,583,17109,26698,0.0160,0.2392,0.2470
7,202510.0,20251001,4887,31894,79856,71520,103414,574,16811,25126,0.0180,0.2351,0.2430
8,202509.0,20250901,4485,27007,74969,71520,98527,543,16460,23534,0.0201,0.2301,0.2389
9,202508.0,20250801,4268,22522,70484,71520,94042,458,16066,21973,0.0203,0.2246,0.2337


In [7]:
df_outcome.to_excel('main_data/evolucion_vinculacion_y_uso_wompi_vinc_primer_regi.xlsx', index=False)

# Eliminación tablas proceso.

In [8]:
# Eliminación tablas proceso.
tablas_borrar = ['proceso.mdo_hist_vinc_wompi_vinc_primer_regi', 'proceso.mdo_hist_vinc_uso_wompi_vinc_primer_regi']

for tabla in tablas_borrar:
    sql_drop = f"""DROP TABLE IF EXISTS {tabla} PURGE;"""
    helper.ejecutar_consulta(sql_drop)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 11/11      DROP ....mdo_hist_vinc_wompi_vinc_primer_regi   finalizado   11:48:53 AM     00:00.6 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 12/12      DROP ..._hist_vinc_uso_wompi_vinc_primer_regi   finalizado   11:48:53 AM     00:00.6 
-------------------------------------------------------------------------------------------------
